In [ ]:
# CLDSA — Cell 0
# Alpaca Session
# 10-March-2026

!pip install alpaca-trade-api yfinance pandas pandas_ta pytz -q

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))
print('Score!')

2026-03-13 17:28:02 IST
Score!


In [ ]:
#Cell 1 — Alpaca Connection
# CLDSA — Cell 1
# Alpaca Connection
# 10-March-2026

from datetime import datetime
import pytz
IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

# Margin never used.
# CLDSA trades cash only.$100,000 paper cash = ceiling.
# $200,000 buying power = ignored.Margin is leverage.
# Leverage multiplies losses as readily as gains.
# In validation phase:margin is forbidden.


import alpaca_trade_api as tradeapi

API_KEY    = 'PKXLOZBC72Q4HL2TCMIOYXMSWY'
SECRET_KEY = 'DNHkgBL8DTcWywzrTf1S6S8Jy5UcniyGcNSBW6U8EPbn'
BASE_URL   = 'https://paper-api.alpaca.markets'

api = tradeapi.REST(
    API_KEY,
    SECRET_KEY,
    BASE_URL,
    api_version='v2'
)

# Add this line before the try block
# to verify what is being sent:

try:
    account = api.get_account()
    print(f"Account status:  {account.status}")
    print(f"Buying power:    ${account.buying_power}")
    print(f"Portfolio value: ${account.portfolio_value}")
    print(f"Paper trading:   YES — no real money")
    print("Score!")
except Exception as e:
    print(f"Connection failed: {e}")

2026-03-13 17:28:02 IST
Account status:  ACTIVE
Buying power:    $200000
Portfolio value: $100000
Paper trading:   YES — no real money
Score!


In [ ]:
# Cell 2 — Alpaca Funds
# CLDSA — Cell 2
# Alpaca Account Funds
# 10-March-2026

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

try:
    account = api.get_account()
    print(f"Cash:             ${account.cash}")
    print(f"Buying power:     ${account.buying_power}")
    print(f"Portfolio value:  ${account.portfolio_value}")
    print(f"Equity:           ${account.equity}")
    print(f"Paper trading:    YES — no real money")
    print("Score!")
except Exception as e:
    print(f"❌ Failed: {e}")

2026-03-13 17:28:02 IST
Cash:             $100000
Buying power:     $200000
Portfolio value:  $100000
Equity:           $100000
Paper trading:    YES — no real money
Score!


In [ ]:
# Cell 3 — Historical Data via yfinance
# CLDSA — Cell 3
# Historical Data — yfinance
# 10-March-2026

from datetime import datetime
import pytz
import yfinance as yf
import pandas as pd

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

# Symbols — yfinance format
# Alpaca uses US markets
SYMBOLS = {
    'AAPL':  'Apple Inc',
    'MSFT':  'Microsoft',
    'GOOGL': 'Alphabet'
}

print("\n===== HISTORICAL DATA — LAST 30 DAYS =====")

for ticker, name in SYMBOLS.items():
    try:
        data = yf.download(
            ticker,
            period='30d',
            interval='1d',
            progress=False
        )
        rows = len(data)
        latest_close = round(data['Close'].iloc[-1].values[0], 2)
        latest_date  = data.index[-1].strftime('%Y-%m-%d')
        print(f"\n{name} ({ticker}):")
        print(f"   Rows fetched:   {rows} days")
        print(f"   Latest close:   ${latest_close}")
        print(f"   Latest date:    {latest_date}")
    except Exception as e:
        print(f"❌ {ticker} failed: {e}")

print("\n==========================================")
print("Score!")

2026-03-13 17:28:03 IST

===== HISTORICAL DATA — LAST 30 DAYS =====

Apple Inc (AAPL):
   Rows fetched:   30 days
   Latest close:   $255.76
   Latest date:    2026-03-12

Microsoft (MSFT):
   Rows fetched:   30 days
   Latest close:   $401.86
   Latest date:    2026-03-12

Alphabet (GOOGL):
   Rows fetched:   30 days
   Latest close:   $303.55
   Latest date:    2026-03-12

Score!


In [ ]:
# Cell 4 — Alpaca Live Prices
# CLDSA — Cell 4
# Live Market Prices — Alpaca
# 10-March-2026

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

# Symbols — US markets
SYMBOLS = {
    'AAPL':  'Apple Inc',
    'MSFT':  'Microsoft',
    'GOOGL': 'Alphabet'
}

print("\n===== LIVE MARKET PRICES — ALPACA =====")

for ticker, name in SYMBOLS.items():
    try:
        trade = api.get_latest_trade(ticker)
        quote = api.get_latest_quote(ticker)
        print(f"\n{name} ({ticker}):")
        print(f"   Last trade:  ${round(trade.price, 2)}")
        print(f"   Bid:         ${round(quote.bid_price, 2)}")
        print(f"   Ask:         ${round(quote.ask_price, 2)}")
    except Exception as e:
        print(f"❌ {ticker} failed: {e}")

fetched_at = datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')
print(f"\n========================================")
print(f"Prices fetched at: {fetched_at}")
print("Score!")

2026-03-13 17:28:04 IST

===== LIVE MARKET PRICES — ALPACA =====

Apple Inc (AAPL):
   Last trade:  $255.71
   Bid:         $245.57
   Ask:         $0

Microsoft (MSFT):
   Last trade:  $402.41
   Bid:         $384.46
   Ask:         $425.2

Alphabet (GOOGL):
   Last trade:  $303.64
   Bid:         $291.34
   Ask:         $316.15

Prices fetched at: 2026-03-13 17:28:06 IST
Score!


In [ ]:
# Cell 4B — Clean Code
# CLDSA — Cell 4B
# Live Price Refresh Loop — Alpaca
# 10-March-2026

import time
from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')

# SYMBOLS carried from Cell 4
# api object carried from Cell 1
# No redefinition needed

print("Live prices — refreshing every 5 seconds")
print("Ctrl+C to stop cleanly")
print("=" * 42)

while True:
    try:
        now = datetime.now(IST).strftime('%H:%M:%S IST')
        print(f"\n{now}")
        for ticker, name in SYMBOLS.items():
            trade = api.get_latest_trade(ticker)
            print(f"  {name:12} ${round(trade.price, 2)}")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\nStopped cleanly. Score!")
        break
    except Exception as e:
        print(f"Error: {e}")
        break

Live prices — refreshing every 5 seconds
Ctrl+C to stop cleanly

17:28:06 IST
  Apple Inc    $255.71
  Microsoft    $402.41
  Alphabet     $303.64

17:28:11 IST
  Apple Inc    $255.71
  Microsoft    $402.41
  Alphabet     $303.64

17:28:17 IST
  Apple Inc    $255.71
  Microsoft    $402.41
  Alphabet     $303.64

Stopped cleanly. Score!


In [ ]:
# Cell 5A — Alpaca/yfinance
# CLDSA — Cell 5A
# Technical Indicators — yfinance + pandas_ta
# 12-March-2026

from datetime import datetime
import pytz
import yfinance as yf
import pandas as pd
import pandas_ta as ta

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

TICKER = 'AAPL'

data = yf.download(TICKER, period='6mo', interval='1d', progress=False)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

data['RSI']    = ta.rsi(data['Close'], length=14)
data['SMA_20'] = ta.sma(data['Close'], length=20)
data['SMA_50'] = ta.sma(data['Close'], length=50)

latest = data.iloc[-1]
rsi    = round(float(latest['RSI']), 2)
sma20  = round(float(latest['SMA_20']), 2)
sma50  = round(float(latest['SMA_50']), 2)
close  = round(float(latest['Close']), 2)
date   = data.index[-1].strftime('%Y-%m-%d')

print(f"\nTicker:  {TICKER}")
print(f"Date:    {date}")
print(f"Close:   ${close}")
print(f"RSI:     {rsi}")
print(f"SMA_20:  ${sma20}")
print(f"SMA_50:  ${sma50}")
print("\nDone With Grace!")

2026-03-13 17:28:21 IST

Ticker:  AAPL
Date:    2026-03-12
Close:   $255.76
RSI:     39.79
SMA_20:  $263.33
SMA_50:  $263.03

Done With Grace!


In [ ]:
# Cell 5B — Signal Engine
# CLDSA — Cell 5B
# Signal Engine — RSI + SMA Logic
# 12-March-2026
# Logic: CLDSA proprietary
# Source: Varsity Module 9 + Chan Ch.3

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

# ─── Risk Parameters (from Risk Doc v1) ───
MAX_POSITION_USD = 1500   # Parameter 1
MAX_SHARES       = 1      # Parameter 2
STOP_LOSS_PCT    = 0.07   # Parameter 3 — 7%
RSI_BUY          = 35     # oversold threshold
RSI_SELL         = 65     # overbought threshold

# ─── Signal Logic ─────────────────────────
def generate_signal(ticker, close, rsi, sma50, buying_power):

    signal   = 'HOLD'
    reason   = ''
    sl_price = None

    # Risk gate first — always
    if buying_power < MAX_POSITION_USD:
        return 'HOLD', 'Risk gate: insufficient buying power', None

    # BUY condition
    if rsi < RSI_BUY and close < sma50:
        signal   = 'BUY'
        reason   = f'RSI {rsi} < {RSI_BUY} AND price ${close} < SMA_50 ${sma50}'
        sl_price = round(close * (1 - STOP_LOSS_PCT), 2)

    # SELL condition
    elif rsi > RSI_SELL and close > sma50:
        signal = 'SELL'
        reason = f'RSI {rsi} > {RSI_SELL} AND price ${close} > SMA_50 ${sma50}'

    else:
        signal = 'HOLD'
        reason = f'RSI {rsi} — no threshold triggered'

    return signal, reason, sl_price

# ─── Run Signal Engine ────────────────────
buying_power = float(api.get_account().buying_power)

print(f"\nBuying power:    ${buying_power}")
print(f"Max per trade:   ${MAX_POSITION_USD}")
print(f"RSI buy <:       {RSI_BUY}")
print(f"RSI sell >:      {RSI_SELL}")
print("\n" + "=" * 45)
print("SIGNAL ENGINE — CLDSA v0.1")
print("=" * 45)

# Uses data from Cell 5A — same session
signal, reason, sl = generate_signal(
    TICKER, close, rsi, sma50, buying_power
)

print(f"\nTicker:   {TICKER}")
print(f"Signal:   {signal}")
print(f"Reason:   {reason}")
if sl:
    print(f"Stop loss if executed: ${sl}")
print(f"\nMax shares: {MAX_SHARES}")
print("\nDone With Grace!")

2026-03-13 17:28:22 IST

Buying power:    $200000.0
Max per trade:   $1500
RSI buy <:       35
RSI sell >:      65

SIGNAL ENGINE — CLDSA v0.1

Ticker:   AAPL
Signal:   HOLD
Reason:   RSI 39.79 — no threshold triggered

Max shares: 1

Done With Grace!


In [ ]:
# Cell 6 — Order Placement
# CLDSA — Cell 6
# Order Placement — Alpaca Paper Trading
# 13-March-2026
# Executes only if Cell 5B signal = BUY
# Paper trading — zero financial risk

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

def place_order(ticker, signal, close_price, buying_power):

    print(f"\n{'='*45}")
    print(f"ORDER ENGINE — CLDSA v0.1")
    print(f"{'='*45}")
    print(f"Signal received: {signal}")
    print(f"Ticker:          {ticker}")
    print(f"Current price:   ${close_price}")
    print(f"Buying power:    ${buying_power}")

    # Risk gate — always first
    if signal != 'BUY':
        print(f"\nNo order placed.")
        print(f"Reason: Signal is {signal} — not BUY")
        print("Done With Grace!")
        return None

    if buying_power < 1500:
        print(f"\nNo order placed.")
        print(f"Reason: Buying power below $1,500")
        print("Done With Grace!")
        return None

    # Place paper order
    try:
        order = api.submit_order(
            symbol=ticker,
            qty=1,
            side='buy',
            type='market',
            time_in_force='day'
        )
        print(f"\n✅ ORDER PLACED")
        print(f"Order ID:    {order.id}")
        print(f"Symbol:      {order.symbol}")
        print(f"Qty:         {order.qty}")
        print(f"Side:        {order.side}")
        print(f"Type:        {order.type}")
        print(f"Status:      {order.status}")
        print(f"Submitted:   {datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')}")
        print("Done With Grace!")
        return order

    except Exception as e:
        print(f"\n❌ Order failed: {e}")
        return None

# Run with Cell 5B values
order = place_order(
    ticker=TICKER,
    signal=signal,
    close_price=close,
    buying_power=float(api.get_account().buying_power)
)

2026-03-13 17:28:22 IST

ORDER ENGINE — CLDSA v0.1
Signal received: HOLD
Ticker:          AAPL
Current price:   $255.76
Buying power:    $200000.0

No order placed.
Reason: Signal is HOLD — not BUY
Done With Grace!


In [ ]:
# Cell 7 — Portfolio Tracking
# CLDSA — Cell 7
# Portfolio Tracking — Alpaca Paper Trading
# 13-March-2026
# Real time positions + PnL
# Point 7 of 7 — original pp1 scope

from datetime import datetime
import pytz

IST = pytz.timezone('Asia/Kolkata')
print(datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST'))

print(f"\n{'='*45}")
print(f"PORTFOLIO TRACKER — CLDSA v0.1")
print(f"{'='*45}")

try:
    # Account summary
    account = api.get_account()
    print(f"\nACCOUNT SUMMARY")
    print(f"Portfolio value: ${float(account.portfolio_value):,.2f}")
    print(f"Cash:            ${float(account.cash):,.2f}")
    print(f"Buying power:    ${float(account.buying_power):,.2f}")
    print(f"Equity:          ${float(account.equity):,.2f}")

    # Open positions
    positions = api.list_positions()
    print(f"\nOPEN POSITIONS")
    if not positions:
        print("No open positions.")
        print("Paper trading — no trades placed yet.")
    else:
        for p in positions:
            pnl      = float(p.unrealized_pl)
            pnl_pct  = float(p.unrealized_plpc) * 100
            pnl_sign = "+" if pnl >= 0 else ""
            print(f"\n  {p.symbol}")
            print(f"  Qty:        {p.qty}")
            print(f"  Entry:      ${float(p.avg_entry_price):,.2f}")
            print(f"  Current:    ${float(p.current_price):,.2f}")
            print(f"  Mkt Value:  ${float(p.market_value):,.2f}")
            print(f"  PnL:        {pnl_sign}${pnl:,.2f} ({pnl_sign}{pnl_pct:.2f}%)")

    # Recent orders
    orders = api.list_orders(status='all', limit=5)
    print(f"\nRECENT ORDERS (last 5)")
    if not orders:
        print("No orders placed yet.")
    else:
        for o in orders:
            print(f"  {o.symbol} | {o.side.upper()} | {o.qty} | {o.status}")

    print(f"\n{'='*45}")
    print(f"Tracked at: {datetime.now(IST).strftime('%Y-%m-%d %H:%M:%S IST')}")
    print("Done With Grace!")

except Exception as e:
    print(f"❌ Failed: {e}")

2026-03-13 17:28:59 IST

PORTFOLIO TRACKER — CLDSA v0.1

ACCOUNT SUMMARY
Portfolio value: $100,000.00
Cash:            $100,000.00
Buying power:    $200,000.00
Equity:          $100,000.00

OPEN POSITIONS
No open positions.
Paper trading — no trades placed yet.

RECENT ORDERS (last 5)
No orders placed yet.

Tracked at: 2026-03-13 17:29:00 IST
Done With Grace!
